In [116]:
import pandas as pd
import numpy as np 
import plotly.express as px
import plotly.graph_objects as go
import math
from plotly.colors import hex_to_rgb
import re
from pathlib import Path

In [117]:
metrics = pd.read_csv("ALL_SPECIES_METRICS_long.tsv", sep="\t")
print(metrics.head())
print(metrics.columns)

                 metric                  label Hydrophis major  \
0       genome_size_Gbp      Genome size (Gbp)            2.15   
1        genome_size_bp         genome_size_bp      2150333220   
2    genome_n_sequences  Sequences in assembly            1111   
3  genome_n_chromosomes      Chromosomes (ch*)               8   
4       genome_chrom_bp        genome_chrom_bp      1987256141   

  Hydrophis curtus (East) Hydrophis curtus (West) Hydrophis cyanocinctus  \
0                   1.923                   2.059                  1.928   
1              1922533803              2058638819             1927585190   
2                     361                      31                     48   
3                       8                       8                      8   
4              1815148967              2056049319             1925852093   

  Hydrophis ornatus  
0             1.998  
1        1997787468  
2               196  
3                 8  
4        1987949638  
Index(['metric

### add `EviAnn` unknown function

```bash
%%bash 
echo "hcy $(awk -F'\t' '$3=="mRNA" && $9~/Note=function unknown/' /hpcfs/users/a1864358/sanders_lab/asm/files/annotation/eviann_results/hcy/hcy.fa.functional_note.pseudo_label.gff | wc -l )"

echo "hcure $(awk -F'\t' '$3=="mRNA" && $9~/Note=function unknown/' /hpcfs/users/a1864358/sanders_lab/asm/files/annotation/eviann_results/hcure/hcure.fa.functional_note.pseudo_label.gff | wc -l )"

echo "hcurw $(awk -F'\t' '$3=="mRNA" && $9~/Note=function unknown/' /hpcfs/users/a1864358/sanders_lab/asm/files/annotation/eviann_results/hcurw/hcurw.fa.functional_note.pseudo_label.gff | wc -l )"

echo "hmaj $(awk -F'\t' '$3=="mRNA" && $9~/Note=function unknown/' /hpcfs/users/a1864358/sanders_lab/asm/files/annotation/eviann_results/hmaj/hmaj.fa.functional_note.pseudo_label.gff | wc -l )"

echo "horn $(awk -F'\t' '$3=="mRNA" && $9~/Note=function unknown/' /hpcfs/users/a1864358/sanders_lab/asm/files/annotation/eviann_results/horn/horn.fa.functional_note.pseudo_label.gff | wc -l )"
```

hcy 4118
hcure 3315
hcurw 4359
hmaj 4415
horn 4173

```python
# add to table
new_row = pd.DataFrame({
    "metric": ["eviann_unknown"],
    "label": ["eviann unknown function"],
    "Hydrophis major": [4415],
    "Hydrophis curtus (East)": [3315],
    "Hydrophis curtus (West)": [4359],
    "Hydrophis cyanocinctus": [4118],
    "Hydrophis ornatus": [4173],
})

metrics = pd.concat([metrics, new_row], ignore_index=True)
metrics.to_csv("ALL_SPECIES_METRICS_long.tsv", sep="\t", index=False)
```

genome_size_bp, genome_chrom_pct, n_gene_total, n_protein_coding, n_processed_pseudogene, n_mRNA, n_CDS, isoforms_per_gene,  genome_n_sequences, agat_pct_mrna_utr_both, agat_mean_mrnas_per_gene, agat_mean_exons_per_gene, agat_mean_cds_length_bp, omark_main_pct , omark_duplicated_pct, omark_inconsisent_pct, omark_unknown_pct, compleasm_C_pct, compleasm_F_pct, compleasm_M_pct, psauron_protein_score, eviann_unknown

- add my annotated pseudogenes

In [118]:
# genome stats
keep = [
    "genome_size_bp", "genome_chrom_pct", "n_gene_total", "n_protein_coding",
    "n_processed_pseudogene", "n_mRNA", "n_CDS", "isoforms_per_gene",
    "genome_n_sequences", "agat_pct_mrna_utr_both", "agat_mean_mrnas_per_gene",
    "agat_mean_exons_per_mrna", "agat_mean_cds_length_bp", 
]

df_stats = metrics[metrics["metric"].isin(keep)]

df_stats_m = df_stats.melt(
    id_vars=["metric", "label"],
    var_name="Species",
    value_name="Count",
)

fig = px.bar(
    df_stats_m,
    x="Species",
    y="Count",
    color="metric",
    barmode="group",
    log_y = True
)

fig.show()

In [119]:
df_stats_m.head()

,metric,label,Species,Count
0,genome_size_bp,genome_size_bp,Hydrophis major,2150333220
1,genome_n_sequences,Sequences in assembly,Hydrophis major,1111
2,genome_chrom_pct,% genome in chromosomes,Hydrophis major,92.42
3,n_gene_total,Total gene features,Hydrophis major,22445
4,n_protein_coding,Protein-coding genes,Hydrophis major,20825


In [120]:
species = [c for c in metrics.columns if c not in ("metric", "label")]


In [121]:
# OMArk completeness
omark = ["omark_single_pct", "omark_duplicated_pct", "omark_missing_pct"]

legend_labels = {
    "omark_single_pct": "Single %",
    "omark_duplicated_pct": "Duplicated %",
    "omark_missing_pct": "Missing %",
}

wide_df = (
    metrics.set_index("metric")[species]
    .loc[omark]
    .astype(float)
    .T
    .rename_axis("Species")
    .reset_index()
)

fig = px.bar(
    wide_df,
    x="Species",
    y=omark,
    text_auto=True,
    barmode = "stack",
    color_discrete_sequence = [
        px.colors.qualitative.Pastel[2],
        px.colors.qualitative.Pastel[0],
        px.colors.qualitative.Pastel[1],
    ]
)

fig.update_traces(
    textposition="inside",
    textfont_color="white",
    textfont_size=14,
    textfont = dict(weight = "bold")
)

species_labels = wide_df["Species"].tolist()

wrapped_species = [
    s.replace(" ", "<br>", 1)   
    for s in species_labels
]

fig.update_xaxes(
    tickmode="array",
    tickvals=species_labels,
    ticktext=wrapped_species,
    tickangle=0,
    tickfont_size=9,
    title=None,
)

fig.update_yaxes(
    range=[92, 100],
    title=None,
    showticklabels=False,
)

fig.update_layout(
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
    ),
    legend_title_text = None,
    margin=dict(t=70),
)

fig.show()

In [122]:
# compleasm
compleasm = ["compleasm_C_pct", "compleasm_F_pct", "compleasm_M_pct"]

legend_labels_busco = {
    "compleasm_C_pct": "Completeness %",
    "compleasm_F_pct": "Fragmented %",
    "compleasm_M_pct": "Missing %",
}

wide_df = (
    metrics.set_index("metric")[species]
    .loc[compleasm]
    .astype(float)
    .T
    .rename_axis("Species")
    .reset_index()
)

fig = px.bar(
    wide_df,
    x="Species",
    y=compleasm,
    text_auto=True,
    barmode = "stack",
    color_discrete_sequence = [
    px.colors.qualitative.Pastel[3],
    px.colors.qualitative.Pastel[5],
    px.colors.qualitative.Pastel[2],
]
)

fig.update_traces(
    textposition="inside",
    textfont_color="white",
    textfont_size=14,
    textfont = dict(weight = "bold")
)

species_labels = wide_df["Species"].tolist()

wrapped_species = [
    s.replace(" ", "<br>", 1)   
    for s in species_labels
]

fig.update_xaxes(
    tickmode="array",
    tickvals=species_labels,
    ticktext=wrapped_species,
    tickangle=0,
    tickfont_size=11,
    title=None,
)

fig.update_yaxes(
    range=[95, 100],
    title=None,
    showticklabels=False,
)

fig.update_layout(
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.05,
        xanchor="center",
        x=0.5,
    ),
    legend_title_text = None,
    margin=dict(t=70),
)


fig.show()

In [123]:
# psauron

psauron = ["PSAURON protein score", "PSAURON CDS score"]

wide_df = (
    metrics.set_index("label")[species]
    .loc[psauron]
    .astype(float)
    .T
    .rename_axis("Species")
    .reset_index()
)

fig = px.bar(
    wide_df,
    x="Species",
    y=psauron,
    text_auto=True,
    barmode = "group",
    log_y = True,
    color_discrete_sequence = [
        px.colors.qualitative.Pastel[6],
        px.colors.qualitative.Pastel[9],
    ]
)

fig.update_traces(
    textposition="inside",
    textfont_color="white",
    textfont_size=14,
    textfont = dict(weight = "bold")
)

species_labels = wide_df["Species"].tolist()

wrapped_species = [
    s.replace(" ", "<br>", 1)   
    for s in species_labels
]

fig.update_xaxes(
    tickmode="array",
    tickvals=species_labels,
    ticktext=wrapped_species,
    tickangle=0,
    tickfont_size=11,
    title=None,
)

fig.update_yaxes(type="log", range=[math.log10(94), math.log10(96.5)])

fig.update_yaxes(
    title=None,
    showticklabels=True,
)

fig.update_layout(
    legend=dict(
        orientation="h",
        yanchor="middle",
        y=0.90,
        xanchor="center",
        x=0.5,
    ),
    legend_title_text = None,
    margin=dict(t=70),
)

fig.show()

In [125]:
# Gene length distributions — Hydrophis Eviann + RefSeq outgroups (protein-coding genes)

EVIANN = Path("/hpcfs/users/a1864358/sanders_lab/asm/files/annotation/eviann_results")
RES = Path("/hpcfs/users/a1864358/sanders_lab/resources")

gffs = {
    "Hydrophis curtus (East)": EVIANN / "hcure/hcure.fa.functional_note.pseudo_label.gff",
    "Hydrophis curtus (West)": EVIANN / "hcurw/hcurw.fa.functional_note.pseudo_label.gff",
    "Hydrophis cyanocinctus": EVIANN / "hcy/hcy.fa.functional_note.pseudo_label.gff",
    "Hydrophis major": EVIANN / "hmaj/hmaj.fa.functional_note.pseudo_label.gff",
    "Hydrophis ornatus": EVIANN / "horn/horn.fa.functional_note.pseudo_label.gff",
    "Crotalus adamanteus": RES / "cadam/genomic.gff",
    "Notechis scutatus": RES / "nscut/genomic.gff",
    "Pseudonaja textilis": RES / "ptext/genomic.gff",
}


def gene_lengths_bp(gff: Path) -> np.ndarray:
    lengths = []
    with gff.open() as fh:
        for line in fh:
            if not line or line.startswith("#"):
                continue
            cols = line.rstrip("\n").split("\t")
            if len(cols) < 9 or cols[2] != "gene":
                continue
            attrs = cols[8]
            if "gene_biotype=" in attrs and "gene_biotype=protein_coding" not in attrs:
                continue
            start, end = float(cols[3]), float(cols[4])
            lengths.append(end - start + 1.0)
    return np.asarray(lengths, dtype=float)


def to_rgb_tuple(color: str):
    if color.startswith("#"):
        h = color.lstrip("#")
        if len(h) == 3:
            h = "".join(ch * 2 for ch in h)
        return tuple(float(int(h[i : i + 2], 16)) for i in (0, 2, 4))
    m = re.fullmatch(r"rgb\((\d+),\s*(\d+),\s*(\d+)\)", color)
    if m:
        return tuple(float(v) for v in m.groups())
    raise ValueError(f"Unrecognized color {color!r}")


lengths_by_sp = {sp: gene_lengths_bp(path).astype(float) for sp, path in gffs.items()}
colors = px.colors.qualitative.Pastel

outgroup_colors = {
    "Pseudonaja textilis": colors[3],
    "Notechis scutatus": colors[6],
    "Crotalus adamanteus": colors[9],
}
hydrophis_palette = [c for i, c in enumerate(colors) if i not in (3, 6, 9)]

nbins = 60
all_log = np.concatenate([np.log10(v.astype(float)) for v in lengths_by_sp.values()]).astype(float)
bins = np.linspace(float(all_log.min()), float(all_log.max()), int(nbins) + 1).astype(float)

fig = go.Figure()
hydro_i = 0

for species, lens in lengths_by_sp.items():
    if species in outgroup_colors:
        c = outgroup_colors[species]
    else:
        c = hydrophis_palette[hydro_i % len(hydrophis_palette)]
        hydro_i += 1
    r, g, b = to_rgb_tuple(c)
    counts, edges = np.histogram(np.log10(lens.astype(float)), bins=bins, density=True)
    counts = counts.astype(float)
    edges = edges.astype(float)
    x = np.repeat(edges, 2)[1:-1].astype(float)
    y = np.repeat(counts, 2).astype(float)
    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode="lines",
            name=f"<i>{species}</i> (n={len(lens):,})",
            line=dict(color=c, width=2.5),
            fill="tozeroy",
            fillcolor=f"rgba({r:.0f},{g:.0f},{b:.0f},0.05)",
        )
    )

fig.update_layout(
    width=800,
    height=900,
    font=dict(family="Times New Roman"),
    plot_bgcolor=px.colors.qualitative.Pastel1[8],
    xaxis_title="log10(gene length, bp)",
    yaxis_title="Density",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5,
        font=dict(family="Times New Roman"),
    ),
    margin=dict(t=80),
    hovermode="x unified",
)
fig.update_xaxes(title_font=dict(family="Times New Roman"))
fig.update_yaxes(title_font=dict(family="Times New Roman"))

fig.show()


In [127]:
# q = px.colors.qualitative.swatches()
# q.show()
# c = px.colors.sequential.swatches_continuous()
# c.show()
# s = px.colors.sequential.swatches()
# s.show()